In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

os.chdir(r"C:\Users\ASUS\Desktop")

img  = cv2.imread("mri.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5,5), 0)

# ── Region Growing ──────────────────────────
def region_growing(img, seed, threshold=15):
    H, W    = img.shape
    visited = np.zeros((H, W), dtype=bool)
    region  = np.zeros((H, W), dtype=np.uint8)
    seed_val = int(img[seed[0], seed[1]])

    queue = deque([seed])
    visited[seed[0], seed[1]] = True

    while queue:
        r, c = queue.popleft()
        region[r, c] = 255

        # شوف الجيران الأربعة
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0<=nr<H and 0<=nc<W and not visited[nr,nc]:
                visited[nr,nc] = True
                # هل هو مشابه للبذرة؟
                if abs(int(img[nr,nc]) - seed_val) < threshold:
                    queue.append((nr, nc))

    return region

# ── اختر نقطة البذرة ──────────────────────
# مركز الصورة تقريباً
H, W   = blur.shape
seed   = (H//2, W//2)
print(f"نقطة البذرة: {seed}")
print(f"قيمة البذرة: {blur[seed[0], seed[1]]}")

# جرب threshold مختلفة
r1 = region_growing(blur, seed, threshold=10)
r2 = region_growing(blur, seed, threshold=20)
r3 = region_growing(blur, seed, threshold=35)

# ── العرض ──────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(gray, cmap='gray')
axes[0].plot(seed[1], seed[0], 'r*', markersize=15)
axes[0].set_title('Original + Seed (★)')

axes[1].imshow(r1, cmap='gray')
axes[1].set_title('Threshold = 10')

axes[2].imshow(r2, cmap='gray')
axes[2].set_title('Threshold = 20')

axes[3].imshow(r3, cmap='gray')
axes[3].set_title('Threshold = 35')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()